# 💳 Ejercicio Integrador — Clasificación: Aprobación de Créditos

**Curso:** Ciencia de Datos  
**Temas:** Modelado · Sesgo y Varianza · Explicabilidad · Equidad

## Contexto

Un banco quiere automatizar la aprobación de préstamos personales.  
El modelo predice si un solicitante va a **incumplir el pago** (`default = 1`) o **pagará correctamente** (`default = 0`).

El banco aprueba el préstamo cuando el modelo predice `default = 0`.

## Variables del dataset

| Variable | Descripción |
|----------|-------------|
| `edad` | Edad en años |
| `ingreso_mensual` | Ingreso mensual en $ |
| `antiguedad_laboral` | Años en el empleo actual |
| `deuda_actual` | Deuda vigente en $ |
| `historial_crediticio` | Score crediticio (300–850) |
| `monto_solicitado` | Monto del préstamo solicitado en $ |
| `cuotas` | Cantidad de cuotas del préstamo |
| `tipo_empleo` | 1 = relación de dependencia, 0 = independiente |
| `genero` | M / F — **variable sensible** |
| `default` | **Target:** 1 = incumplió, 0 = pagó correctamente |

---
## Parte 0 — Setup y Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 110

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

In [ ]:
def bloque(rng, n_z, genero_label, ingreso_mu, genero_bonus):
    edad             = rng.integers(22, 65, n_z).astype(float)
    ingreso          = rng.normal(ingreso_mu, 15000, n_z).clip(8000, 250000)
    antiguedad_lab   = rng.uniform(0, 20, n_z).round(1)
    deuda_actual     = rng.normal(35000, 18000, n_z).clip(0, 150000)
    historial        = rng.normal(620, 70, n_z).clip(300, 850).round()
    monto_solicitado = rng.normal(55000, 20000, n_z).clip(5000, 200000)
    cuotas           = rng.integers(6, 61, n_z).astype(float)
    tipo_empleo      = rng.binomial(1, 0.6, n_z).astype(float)
    genero           = np.array([genero_label] * n_z)

    logit = (- 0.022*(historial-600) + 0.000014*deuda_actual
             - 0.000006*ingreso - 0.30*tipo_empleo
             + 0.008*cuotas + 0.5 + genero_bonus + rng.normal(0, 0.25, n_z))
    prob_default = 1 / (1 + np.exp(-logit))
    default = rng.binomial(1, prob_default, n_z)

    return pd.DataFrame({'edad': edad, 'ingreso_mensual': ingreso,
        'antiguedad_laboral': antiguedad_lab, 'deuda_actual': deuda_actual,
        'historial_crediticio': historial, 'monto_solicitado': monto_solicitado,
        'cuotas': cuotas, 'tipo_empleo': tipo_empleo,
        'genero': genero, 'default': default})

def generar_dataset(n=1000, seed=42):
    rng = np.random.default_rng(seed)
    n_m, n_f = n//2, n - n//2
    df = pd.concat([
        bloque(rng, n_m, 'M', ingreso_mu=88000, genero_bonus= 0.0),
        bloque(rng, n_f, 'F', ingreso_mu=60000, genero_bonus=-1.0),
    ], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df = generar_dataset()

# Género como feature — one-hot encoding
df_enc = pd.get_dummies(df, columns=['genero'], drop_first=False)
features = ['edad', 'ingreso_mensual', 'antiguedad_laboral', 'deuda_actual',
            'historial_crediticio', 'monto_solicitado', 'cuotas', 'tipo_empleo',
            'genero_M', 'genero_F']

X = df_enc[features]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
genero_test = df.loc[y_test.index, 'genero'].values

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Dataset: {df.shape}  |  Default global: {y.mean():.3f}")
print(df.groupby('genero')['default'].agg(['sum','count','mean']).round(3))

### 📋 Consigna 0.1 — Análisis exploratorio

a. ¿Cuál es la tasa de default por género? ¿Quién defaultea más?  
b. ¿Difiere el ingreso mensual entre géneros?  
c. Calcular la tasa de default por tramo de ingreso y por género. ¿A igual ingreso, quién defaultea más?  
d. ¿Qué hipótesis genera ese resultado sobre el comportamiento del modelo?

In [ ]:
colores = {'M': 'steelblue', 'F': 'tomato'}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# TODO: a. Barras de tasa de default por género en axes[0]

# TODO: b. Boxplot de ingreso_mensual por género en axes[1]

# TODO: c. Barras de default por tramo de ingreso × género en axes[2]
#          Usar sns.barplot con hue='genero'
tramos = pd.cut(df['ingreso_mensual'],
                bins=[0, 40000, 60000, 80000, 100000, 999999],
                labels=['<40k','40-60k','60-80k','80-100k','>100k'])
tabla  = df.groupby([tramos,'genero'])['default'].mean().unstack('genero')

plt.tight_layout()
plt.show()

print("\nDefault por tramo de ingreso × género:")
print(tabla.round(3).to_string())

**✏️ Interpretación:**  
*(Responder las preguntas a-d)*

---
## Parte 1 — Modelado

### Métricas de clasificación relevantes

En crédito, los errores no tienen el mismo costo:

| Error | Descripción | Consecuencia |
|-------|-------------|-------------|
| **Falso Positivo** (FP) | Predice default, pero pagaría | Se rechaza un buen cliente |
| **Falso Negativo** (FN) | Predice no-default, pero incumple | Se aprueba un mal pagador |

$$\text{FNR} = \frac{FN}{FN + TP} \qquad \text{FPR} = \frac{FP}{FP + TN} \qquad \text{Tasa de aprobación} = \frac{TN + FP}{N}$$

### 📋 Consigna 1.1 — Entrenar y comparar modelos

Entrenar los tres modelos y completar la tabla:

| Modelo | Accuracy | Precision | Recall | F1 | ROC-AUC | Gap Acc |
|--------|----------|-----------|--------|----|---------|---------|
| Reg. Logística | | | | | | |
| Árbol (depth=5) | | | | | | |
| Random Forest | | | | | | |

- ¿Qué modelo tiene mejor ROC-AUC?
- ¿Cuál tiene mayor gap entre accuracy de train y test? ¿Qué indica eso?

In [ ]:
modelos = {
    'Reg. Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Árbol (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':   RandomForestClassifier(n_estimators=100, random_state=42),
}

# TODO: entrenar cada modelo
# Nota: usar X_train_sc / X_test_sc para Reg. Logística, X_train / X_test para los árboles
# TODO: calcular métricas en train y test
# TODO: mostrar tabla comparativa

In [ ]:
# TODO: graficar matrices de confusión para los tres modelos (1 fila, 3 columnas)
# Usar ConfusionMatrixDisplay

**✏️ Interpretación:**  
*(¿Qué modelo elegirías considerando el costo asimétrico de los errores?)*

---
## Parte 2 — Sesgo y Varianza

### 📋 Consigna 2.1 — Efecto de la profundidad del árbol

Para el árbol de decisión, calcular accuracy en train y test para `max_depth` ∈ {1, 2, 3, 5, 7, 10, 15, None}.

- Graficar las dos curvas (train y test) en función de la profundidad.
- ¿En qué profundidad se maximiza el test accuracy?
- ¿A partir de qué profundidad el modelo claramente overfittea?
- ¿Cómo se relaciona esto con los conceptos de sesgo y varianza?

In [ ]:
profundidades = [1, 2, 3, 5, 7, 10, 15, None]

# TODO: para cada profundidad, entrenar árbol, calcular accuracy train y test
# TODO: graficar curvas train/test vs profundidad
# TODO: marcar la profundidad óptima con una línea vertical

**✏️ Interpretación:**  
*(¿Dónde está el punto de balance entre bias y varianza?)*

### 📋 Consigna 2.2 — Estabilidad del modelo

Entrenar el árbol (depth=5) y el Random Forest **10 veces** con distintas semillas.  
Calcular ROC-AUC en cada caso.

- ¿Cuál modelo es más estable?
- ¿Cómo se relaciona la estabilidad con el concepto de varianza del estimador?
- ¿Por qué importa la estabilidad en un sistema de decisiones crediticias?

In [ ]:
# TODO: para cada seed en range(10):
#   - train_test_split con random_state=seed
#   - entrenar árbol (depth=5) y RF
#   - calcular ROC-AUC en test
# TODO: imprimir media y std de cada modelo
# TODO: boxplot comparativo

**✏️ Interpretación:**  
*(¿Qué consecuencias tendría en producción un modelo con alta varianza del estimador?)*

---
## Parte 3 — Explicabilidad con SHAP

> En clasificación binaria, `TreeExplainer` devuelve SHAP values para las dos clases.  
> Seleccionamos la clase 1 (default) con `[:, :, 1]`.  
> Cada $\phi_j > 0$ significa que esa feature **aumenta** la probabilidad de default.

In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
try:
    import shap
    from packaging.version import Version
    if Version(shap.__version__) < Version('0.46'): raise ImportError
except ImportError:
    install('shap>=0.46'); import shap

rf          = modelos['Random Forest']
explainer   = shap.TreeExplainer(rf)
shap_values = explainer(X_test)[:, :, 1]   # clase 1 = default
preds_test  = rf.predict(X_test)
proba_test  = rf.predict_proba(X_test)[:, 1]

print(f"shap_values shape: {shap_values.values.shape}")

### 📋 Consigna 3.1 — Importancia global

a. Graficar el bar plot de importancia SHAP.  
b. ¿Qué feature tiene mayor impacto promedio sobre la probabilidad de default?  
c. ¿`genero_M` y `genero_F` aparecen en el ranking? ¿Qué implica eso?

In [ ]:
# TODO: shap.plots.bar

**✏️ Interpretación:**  
*(¿Qué variables dominan? ¿Qué dice el ranking sobre el rol del género?)*

### 📋 Consigna 3.2 — Beeswarm

a. Graficar el beeswarm plot.  
b. ¿Cómo afecta `historial_crediticio` a la probabilidad de default? ¿Y `deuda_actual`?  
c. Para `genero_M` y `genero_F`: ¿en qué dirección empujan la predicción? ¿Es consistente con lo que viste en el EDA?

In [ ]:
# TODO: shap.plots.beeswarm

**✏️ Interpretación:**  
*(Describir el efecto de al menos 3 variables, incluyendo género)*

### 📋 Consigna 3.3 — Dos casos contrastantes

Comparar:
- **Caso A:** una mujer a quien el modelo predice default (prob > 0.65)
- **Caso B:** un hombre con ingreso similar al caso A pero aprobado (prob < 0.40)

Para cada caso mostrar el waterfall plot e identificar qué features inclinaron la decisión.  
¿Qué rol juega el género en cada caso?

In [ ]:
# Caso A: mujer con predicción alta de default
mask_f_alto = (genero_test == 'F') & (proba_test > 0.65)
idx_a = np.where(mask_f_alto)[0][0]

# Caso B: hombre con ingreso similar al caso A pero aprobado
ing_a = X_test.iloc[idx_a]['ingreso_mensual']
mask_m_aprob = ((genero_test == 'M') & (proba_test < 0.40) &
                (np.abs(X_test['ingreso_mensual'].values - ing_a) < 15000))
idx_b = np.where(mask_m_aprob)[0][0]

# TODO: mostrar características de ambos casos
# TODO: shap.plots.waterfall para idx_a
# TODO: shap.plots.waterfall para idx_b

**✏️ Interpretación:**  
*(¿La diferencia en la decisión se explica por el género o por otras variables?)*

---
## Parte 4 — Equidad (Fairness)

### Métricas de equidad

| Métrica | Definición | Pregunta que responde |
|---------|------------|----------------------|
| **Demographic Parity** | Tasa de aprobación igual por grupo | ¿El banco aprueba en la misma proporción? |
| **Equal Opportunity** | FNR igual por grupo | ¿Los que defaultean son detectados igual en cada grupo? |

### 📋 Consigna 4.1 — Métricas por género

Calcular para cada género: N, default real, tasa de aprobación, accuracy, FNR, FPR.

- ¿Se cumple Demographic Parity?
- ¿Se cumple Equal Opportunity?
- ¿El grupo con menor default real recibe mayor o menor aprobación?

In [ ]:
# TODO: calcular métricas por género (M y F)
# TODO: mostrar tabla comparativa
# TODO: graficar barras dobles: default real, tasa aprobación, FNR, FPR

**✏️ Interpretación:**  
*(¿Se cumple alguna noción de equidad? ¿La diferencia en aprobación refleja el riesgo real?)*

### 📋 Consigna 4.2 — ¿Incluir género mejora la equidad?

Entrenar el mismo Random Forest **sin** las variables `genero_M` y `genero_F`.

Comparar con el modelo original:

| Modelo | AUC | AprobM | AprobF | FNR_M | FNR_F |
|--------|-----|--------|--------|-------|-------|
| Con género | | | | | |
| Sin género | | | | | |

- ¿Cambia el AUC al incluir género?
- ¿Cambian las tasas de aprobación?
- ¿Qué conclusión sacás sobre el rol del ingreso como variable correlacionada con el género?

In [ ]:
features_sin = ['edad', 'ingreso_mensual', 'antiguedad_laboral', 'deuda_actual',
                'historial_crediticio', 'monto_solicitado', 'cuotas', 'tipo_empleo']

# TODO: entrenar RF sin género
# TODO: calcular métricas por género para ambos modelos
# TODO: comparar tabla

**✏️ Interpretación:**  
*(¿Excluir el género del modelo elimina el sesgo? ¿Por qué?)*

---
## Parte 5 — Análisis con librerías especializadas

### 5.1 — Sesgo y varianza con `mlxtend`

`bias_variance_decomp` entrena el modelo muchas veces en subsets del training set
y estima empíricamente las tres componentes del error:

$$\text{Error total} \approx \text{Bias}^2 + \text{Varianza}$$

### 📋 Consigna 5.1

Calcular bias y varianza para el árbol (depth=5) y el Random Forest usando `bias_variance_decomp`.  
Comparar con lo observado manualmente en la consigna 2.2. ¿Qué componente reduce el RF?

In [ ]:
!pip install mlxtend -q

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

modelos_bv = {
    'Árbol (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':   RandomForestClassifier(n_estimators=100, random_state=42),
}

# TODO: para cada modelo, calcular bias_variance_decomp con loss='0-1_loss'
# Nota: pasar X_train.values, y_train.values, X_test.values, y_test.values
# TODO: mostrar tabla Loss / Bias / Varianza

**✏️ Interpretación:**  
*(¿Qué componente del error reduce el RF respecto al árbol?)*

### 5.2 — Equidad con `dalex`

`dalex` calcula automáticamente las métricas de equidad y las compara contra un grupo de referencia.
Para cada métrica calcula el **ratio** grupo / grupo de referencia.
Si el ratio cae fuera del intervalo $(\epsilon, 1/\epsilon)$ con $\epsilon=0.8$, detecta sesgo.

| Métrica | Nombre completo | Descripción |
|---------|----------------|-------------|
| **TPR** | True Positive Rate | De los que defaultean, ¿los detecta igual en cada grupo? |
| **ACC** | Accuracy | Precisión global por grupo |
| **PPV** | Positive Predictive Value | Cuando predice default, ¿acierta igual en cada grupo? |
| **FPR** | False Positive Rate | Buenos clientes rechazados por grupo |
| **STP** | Statistical Parity | Tasa de aprobación por grupo |

### 📋 Consigna 5.2

Usar `dalex` para analizar la equidad del Random Forest con `genero` como variable sensible.  
- ¿Qué métricas detecta como sesgadas?
- ¿Es consistente con lo que calculaste manualmente en la consigna 4.1?

In [ ]:
!pip install dalex -q

In [ ]:
import dalex as dx

rf_dalex = modelos['Random Forest']
exp = dx.Explainer(rf_dalex, X_test.copy(), y_test.copy(), verbose=False)

# TODO: definir protected y privileged
# Hint: protected = np.where(genero_test == 'F', 'Female', 'Male')
#        privileged = 'Male'
# TODO: exp.model_fairness(...)
# TODO: fobject.fairness_check(epsilon=0.8)
# TODO: fobject.plot()

**✏️ Interpretación:**  
*(¿Qué métrica está en zona roja? ¿Qué significa en el contexto del crédito bancario?)*